# Chapter 9: From Predictions to Potential Profit

This notebook mirrors the revised Chapter 9 flow. We first load the live Football-Data.co.uk CSV, then compare three side-selection rules using only confusion matrices and accuracy. After that, we layer fixed-stake and Kelly staking on top of two simple rules: always home and lowest odds.


## What This Notebook Covers

- loading the live `E0.csv` match-and-odds file directly from its URL
- creating a simple chronological train/validation split
- evaluating three side-selection rules
- using confusion matrices and accuracy only for the side-selection comparison
- applying fixed-stake betting to always-home and lowest-odds rules
- applying Kelly staking to the same two rules
- leaving away-team, draw, and highest-odds variants as exercises


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data_url = "https://www.football-data.co.uk/mmz4281/2526/E0.csv"
matches_df = pd.read_csv(data_url)

keep_columns = [
    "Date", "Time", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR", "B365H", "B365D", "B365A",
]
matches_df = matches_df[keep_columns].dropna().copy()

matches_df["Kickoff"] = pd.to_datetime(
    matches_df["Date"] + " " + matches_df["Time"],
    format="%d/%m/%Y %H:%M",
)
matches_df = matches_df.sort_values("Kickoff").reset_index(drop=True)

for outcome in ["H", "D", "A"]:
    matches_df[f"Imp{outcome}"] = 1 / matches_df[f"B365{outcome}"]
matches_df["Overround"] = matches_df[["ImpH", "ImpD", "ImpA"]].sum(axis=1)

cutoff = int(len(matches_df) * 2 / 3)
train_df = matches_df.iloc[:cutoff].copy()
valid_df = matches_df.iloc[cutoff:].copy()

odds_to_result = {"B365H": "H", "B365D": "D", "B365A": "A"}
for frame in [train_df, valid_df]:
    frame["home_pick"] = "H"
    frame["lowest_odds_pick"] = (
        frame[["B365H", "B365D", "B365A"]]
        .idxmin(axis=1)
        .map(odds_to_result)
    )

def confusion_table(actual, predicted):
    return pd.crosstab(
        actual,
        predicted,
        rownames=["Actual"],
        colnames=["Predicted"],
        dropna=False,
    )

print(f"Matches loaded: {len(matches_df)}")
print(f"Train matches: {len(train_df)} | Validation matches: {len(valid_df)}")


## Minimal Dataset Preview


In [ ]:
matches_df[keep_columns].head(10)


## Choosing a Side

This is the first decision layer. We compare three ways to choose the outcome class and evaluate them only with confusion matrices and accuracy.


In [ ]:
home_accuracy = (valid_df["home_pick"] == valid_df["FTR"]).mean()
home_confusion = confusion_table(valid_df["FTR"], valid_df["home_pick"])

print(f"Always-home accuracy: {home_accuracy:.3f}")
home_confusion


In [ ]:
lowest_accuracy = (valid_df["lowest_odds_pick"] == valid_df["FTR"]).mean()
lowest_confusion = confusion_table(valid_df["FTR"], valid_df["lowest_odds_pick"])

print(f"Lowest-odds accuracy: {lowest_accuracy:.3f}")
lowest_confusion


In [ ]:
features = ["HomeTeam", "AwayTeam", "ImpH", "ImpD", "ImpA", "Overround"]

preprocess = ColumnTransformer(
    [
        ("teams", OneHotEncoder(handle_unknown="ignore"), ["HomeTeam", "AwayTeam"]),
        ("numeric", StandardScaler(), ["ImpH", "ImpD", "ImpA", "Overround"]),
    ]
)

ml_model = Pipeline(
    [
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=2000, solver="lbfgs")),
    ]
)

ml_model.fit(train_df[features], train_df["FTR"])
valid_df["ml_pick"] = ml_model.predict(valid_df[features])

ml_accuracy = (valid_df["ml_pick"] == valid_df["FTR"]).mean()
ml_confusion = confusion_table(valid_df["FTR"], valid_df["ml_pick"])

print(f"Machine-learning accuracy: {ml_accuracy:.3f}")
ml_confusion


In [ ]:
comparison = pd.DataFrame(
    {
        "strategy": ["always_home", "lowest_odds", "machine_learning"],
        "accuracy": [home_accuracy, lowest_accuracy, ml_accuracy],
    }
).sort_values("accuracy", ascending=False).reset_index(drop=True)

comparison


## Betting Strategies

Now we keep the side-selection rule fixed and change only the staking rule. The functions below implement flat staking and Kelly staking so we can apply them to the always-home and lowest-odds rules.


In [ ]:
def fixed_stake_summary(matches, picks, stake=100):
    total_profit = 0.0
    for pick, (_, row) in zip(picks, matches.iterrows()):
        odds = row[f"B365{pick}"]
        profit = stake * (odds - 1) if row["FTR"] == pick else -stake
        total_profit += profit

    total_staked = stake * len(matches)
    roi = total_profit / total_staked if total_staked else 0.0
    return total_profit, roi


def kelly_fraction(probability, decimal_odds):
    b = decimal_odds - 1
    q = 1 - probability
    if b <= 0:
        return 0.0
    return max((b * probability - q) / b, 0.0)


def kelly_summary(matches, picks, probabilities, starting_bankroll=1000):
    bankroll = starting_bankroll

    for pick, probability, (_, row) in zip(picks, probabilities, matches.iterrows()):
        odds = row[f"B365{pick}"]
        fraction = min(kelly_fraction(probability, odds), 1.0)
        stake = bankroll * fraction
        bankroll -= stake

        if row["FTR"] == pick:
            bankroll += stake * odds

    roi = (bankroll - starting_bankroll) / starting_bankroll if starting_bankroll else 0.0
    return bankroll, roi


In [ ]:
home_fixed_profit, home_fixed_roi = fixed_stake_summary(
    valid_df,
    valid_df["home_pick"],
    stake=100,
)

lowest_fixed_profit, lowest_fixed_roi = fixed_stake_summary(
    valid_df,
    valid_df["lowest_odds_pick"],
    stake=100,
)

pd.DataFrame(
    {
        "rule": ["always_home", "lowest_odds"],
        "profit": [home_fixed_profit, lowest_fixed_profit],
        "roi": [home_fixed_roi, lowest_fixed_roi],
    }
)


In [ ]:
home_rule_probability = (train_df["FTR"] == "H").mean()
lowest_rule_probability = (train_df["lowest_odds_pick"] == train_df["FTR"]).mean()

home_kelly_bankroll, home_kelly_roi = kelly_summary(
    valid_df,
    valid_df["home_pick"],
    [home_rule_probability] * len(valid_df),
    starting_bankroll=1000,
)

lowest_kelly_bankroll, lowest_kelly_roi = kelly_summary(
    valid_df,
    valid_df["lowest_odds_pick"],
    [lowest_rule_probability] * len(valid_df),
    starting_bankroll=1000,
)

pd.DataFrame(
    {
        "rule": ["always_home", "lowest_odds"],
        "final_bankroll": [home_kelly_bankroll, lowest_kelly_bankroll],
        "roi": [home_kelly_roi, lowest_kelly_roi],
    }
)


## ROI Time Series for the Home-Team Rule

The plot below compares a fixed 100-unit stake with Kelly staking when the chosen side is always the home team. To keep the comparison fair, both paths start from the same bankroll: `100 * len(valid_df)`.


In [ ]:
import matplotlib.pyplot as plt

def fixed_stake_bankroll_path(matches, picks, stake=100, starting_bankroll=None):
    if starting_bankroll is None:
        starting_bankroll = stake * len(matches)
    bankroll = starting_bankroll
    roi_path = []
    for pick, (_, row) in zip(picks, matches.iterrows()):
        odds = row[f"B365{pick}"]
        bankroll -= stake
        if row["FTR"] == pick:
            bankroll += stake * odds
        roi_path.append((bankroll - starting_bankroll) / starting_bankroll if starting_bankroll else 0.0)
    return roi_path

def kelly_bankroll_path(matches, picks, probabilities, starting_bankroll=1000):
    bankroll = starting_bankroll
    roi_path = []
    for pick, probability, (_, row) in zip(picks, probabilities, matches.iterrows()):
        odds = row[f"B365{pick}"]
        fraction = min(kelly_fraction(probability, odds), 1.0)
        stake = bankroll * fraction
        bankroll -= stake
        if row["FTR"] == pick:
            bankroll += stake * odds
        roi_path.append((bankroll - starting_bankroll) / starting_bankroll if starting_bankroll else 0.0)
    return roi_path

comparison_bankroll = 100 * len(valid_df)
home_fixed_roi_path = fixed_stake_bankroll_path(
    valid_df, valid_df["home_pick"], stake=100, starting_bankroll=comparison_bankroll
)
home_kelly_roi_path = kelly_bankroll_path(
    valid_df,
    valid_df["home_pick"],
    [home_rule_probability] * len(valid_df),
    starting_bankroll=comparison_bankroll,
)

plt.figure(figsize=(11, 5))
plt.plot(valid_df["Kickoff"], home_fixed_roi_path, label="Fixed Stake, Always Home", linewidth=2)
plt.plot(valid_df["Kickoff"], home_kelly_roi_path, label="Kelly, Always Home", linewidth=2)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("Kickoff Date")
plt.ylabel("ROI")
plt.title("ROI Time Series for Fixed Stake and Kelly on the Home-Team Rule")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


## Exercise

Repeat the fixed-stake and Kelly analysis for other side-selection rules such as always away, always draw, or always highest odds. Compare what changes in both prediction accuracy and bankroll outcomes once you change the rule used to choose the side.


## Optional Local Script

If you also want a reusable local script version of this workflow, the synced helper now lives at `Companion-Code/extras/chapter-9/real_data_betting_workflow.py`. It follows the same structure as this notebook and writes a small preview CSV plus a summary JSON.
